# Opdracht 1 · Verkenning & pairplot
### Classificatie-training · Antwoordmodel-versie

In deze opdracht maak je kennis met de Iris-dataset en de basis van classificatie. Je gaat:

1. De Iris-dataset laden en de eerste rijen bekijken
2. De **klassebalans** checken (hoeveel bloemen per soort?)
3. Een **seaborn pairplot** maken om de klassen te verkennen
4. Een **gestratificeerde train/test-split** maken
5. Een **baseline** bouwen (altijd de meerderheidsklasse voorspellen) en de accuracy meten

> **Hoe werkt dit notebook?** De meeste cellen zijn ingevuld; op de plekken met **`# TODO`** vul jij iets in.


## 0. Voorbereiding
Imports. Gewoon uitvoeren.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

sns.set_theme(style="whitegrid")
print("Bibliotheken geladen.")

## 1. De Iris-dataset laden
Iris zit standaard in scikit-learn (geen download nodig). 150 bloemen, drie soorten, vier metingen per bloem.

In [ ]:
data = load_iris(as_frame=True)
df = data.frame.copy()
df["soort"] = df["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
df.head()

## 2. Klassebalans checken
Bij classificatie is het altijd verstandig om eerst te kijken hoe de klassen verdeeld zijn.

### TODO 1 — Tel het aantal bloemen per soort
Gebruik `.value_counts()` op de kolom `"soort"`.

> Tip: `df["soort"].value_counts()`.

In [ ]:
# ANTWOORD
print(df["soort"].value_counts())

Bij Iris is de balans perfect (50/50/50). In de praktijk is dat zelden zo —
zie de slides over klasse-onbalans.

## 3. Seaborn pairplot
Een pairplot zet alle features paarsgewijs tegen elkaar. In één plaatje zie je hoe goed de klassen
al van elkaar te scheiden zijn.

### TODO 2 — Maak de pairplot
Gebruik `sns.pairplot` op de hele dataframe (zonder de target-kolom).
Kleur de punten op `"soort"`.

> Tip: `sns.pairplot(df.drop(columns=["target"]), hue="soort")`.

In [ ]:
# ANTWOORD
sns.pairplot(df.drop(columns=["target"]), hue="soort", height=2)
plt.show()

> **Bekijk de pairplot:** welke soort scheidt zich duidelijk af? Welke twee soorten overlappen?
> Welke twee features lijken het meest scheidend?

## 4. Train/test-split met stratificatie
We splitsen in 80% train en 20% test. Met `stratify=y` houden we de klassebalans in beide sets gelijk —
belangrijk bij classificatie.

### TODO 3 — Splits target en features en maak de gestratificeerde split
Zet `y` op de soort-kolom en `X` op de vier metingen. Gebruik `train_test_split` met `stratify=y`,
`test_size=0.2` en `random_state=42`.

> Tip: de feature-kolommen zijn `data.feature_names`.

In [ ]:
# ANTWOORD
y = df["soort"]
X = df[data.feature_names]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
print("Klassebalans in test:")
print(y_test.value_counts())

## 5. Een baseline bouwen
Bij classificatie is de simpele baseline: altijd de meerderheidsklasse voorspellen.
Een echt model moet deze baseline duidelijk verslaan.

### TODO 4 — Train een DummyClassifier en meet de accuracy
Maak een `DummyClassifier` met `strategy="most_frequent"`, train op de train-set, voorspel de test-set,
en bereken de accuracy.

> Tip: `DummyClassifier(strategy="most_frequent")`.

In [ ]:
# ANTWOORD
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Baseline accuracy: {acc:.3f}")

## 6. Bespreking
Bespreek met je buur:

- Welke accuracy haalde de baseline? Past dat bij de klassebalans van Iris?
- Welke twee features lijken in de pairplot het meest scheidend?
- Welke twee soorten zullen straks het lastigst van elkaar te onderscheiden zijn?
- Waarom is `stratify=y` belangrijk bij classificatie?

---
### Toelichting voor de trainer
- Verwacht **baseline accuracy ≈ 0.33** (één van de drie klassen). Bij sterk onbalans zou de baseline
  veel hoger scoren — leg uit dat dat misleidend is bij ongebalanceerde data.
- In de pairplot is **setosa** perfect scheidbaar; **versicolor** en **virginica** overlappen,
  vooral op kelkblad-metingen. Bloemblad-lengte en -breedte zijn het sterkst scheidend.
- Stratificatie is hier minder kritiek (gebalanceerde data), maar belangrijk om als gewoonte
  aan te leren — bij churn/spam is het wel essentieel.